# Welcome to the Day 2 Lab!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Just before we get started --</h2>
            <span style="color:#f71;">I thought I'd take a second to point you at this page of useful resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

## First - let's talk about the Chat Completions API

1. The simplest way to call an LLM
2. It's called Chat Completions because it's saying: "here is a conversation, please predict what should come next"
3. The Chat Completions API was invented by OpenAI, but it's so popular that everybody uses it!

### We will start by calling OpenAI again - but don't worry non-OpenAI people, your time is coming!


In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


## Do you know what an Endpoint is?

If not, please review the Technical Foundations guide in the guides folder

And, here is an endpoint that might interest you...

In [3]:

import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

{'model': 'gpt-5-nano',
 'messages': [{'role': 'user', 'content': 'Tell me a fun fact'}]}

As you can see above we just have a JSON object being encoded.

Below we can see the chat.completions.create (wow) but in web form.

In [ ]:
# but now we need to parse this dirty response
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

{'id': 'chatcmpl-DJo7TUKHXABpF3yx79jboUGF5m9kN',
 'object': 'chat.completion',
 'created': 1773613263,
 'model': 'gpt-5-nano-2025-08-07',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Fun fact: Honey never spoils. Archaeologists have found pots of honey in ancient Egyptian tombs that are over 3,000 years old and still edible when kept sealed.',
    'refusal': None,
    'annotations': []},
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 11,
  'completion_tokens': 941,
  'total_tokens': 952,
  'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
  'completion_tokens_details': {'reasoning_tokens': 896,
   'audio_tokens': 0,
   'accepted_prediction_tokens': 0,
   'rejected_prediction_tokens': 0}},
 'service_tier': 'default',
 'system_fingerprint': None}

In [5]:
response.json()["choices"][0]["message"]["content"]

'Fun fact: Honey never spoils. Archaeologists have found pots of honey in ancient Egyptian tombs that are over 3,000 years old and still edible when kept sealed.'

# What is the openai package?

It's known as a Python Client Library.

It's nothing more than a wrapper around making this exact call to the http endpoint.

It just allows you to work with nice Python code instead of messing around with janky json objects.

But that's it. It's open-source and lightweight. Some people think it contains OpenAI model code - it doesn't! It's jsut an API wrapper for the chat.completions endpoit we did above. And as you can see below


In [4]:
# Create OpenAI client

from openai import OpenAI
openai = OpenAI()

# now we have the model and message (which is the payload)

response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content



'Fun fact: Wombat poop is cube-shaped. The cube shape helps it stack without rolling away on rough terrain, and it’s formed by the wombat’s unique intestinal muscles and dehydration. Want another fun fact?'

## And then this great thing happened:

OpenAI's Chat Completions API was so popular, that the other model providers created endpoints that are identical.

They are known as the "OpenAI Compatible Endpoints".

For example, google made one here: https://generativelanguage.googleapis.com/v1beta/openai/

And OpenAI decided to be kind: they said, hey, you can just use the same client library that we made for GPT. We'll allow you to specify a different endpoint URL and a different key, to use another provider.

So you can use:

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

And to be clear - even though OpenAI is in the code, we're only using this lightweight python client library to call the endpoint - there's no OpenAI model involved here.

If you're confused, please review Guide 9 in the Guides folder!

And now let's try it!

## THIS IS OPTIONAL - but if you wish to try out Google Gemini, please visit:

https://aistudio.google.com/

And set up your API key at

https://aistudio.google.com/api-keys

And then add your key to the `.env` file, being sure to Save the .env file after you change it:

`GOOGLE_API_KEY=AIz...`


In [5]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")



API key found and looks good so far!


In [6]:
from IPython.display import display, Markdown # imported to help with formatting the output of Gemini responses in a more readable way

gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key) # as you can see here we actually have to specify the base url for Gemini, since it's different than the OpenAI one - but we can still use the same OpenAI client!

response = gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=[{"role": "user", "content": "Tell me a funny fun fact"}])

content = response.choices[0].message.content

display(Markdown(content))

Here's a funny fun fact for you:

**Cows have best friends, and they get stressed out when they are separated.**

Imagine a cow nudging its BFF, "Ugh, Brenda, where did you go? Did you hear what Daisy said about my mooing? So rude!"

## And Ollama also gives an OpenAI compatible endpoint

...and it's on your local machine!

If the next cell doesn't print "Ollama is running" then please open a terminal and run `ollama serve`

In [ ]:
requests.get("http://localhost:11434").content

# becuase we can check this through a browser we just call a simple request

b'Ollama is running'

### Download llama3.2 from meta

Change this to llama3.2:1b if your computer is smaller.

Don't use llama3.3 or llama4! They are too big for your computer..

the ! is to run this as a terminal command within Juypter

In [13]:
!ollama pull llama3.2

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [ ]:
# just look up MODEL + base url and it will get us there
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='whatever_we_want')



In [ ]:
# helps to find the models we have downloaded and can use locally
! ollama list

]11;?\NAME                    ID              SIZE      MODIFIED       
glm-4.7-flash:latest    4475827791a2    19 GB     5 minutes ago     
llama3.2:latest         a80c4f17acd5    2.0 GB    17 minutes ago    
deepseek-r1:1.5b        e0979632db5a    1.1 GB    2 months ago      


In [ ]:
# Get a fun fact, see how systems prompts and user prompts work, and how we can use the same OpenAI client to interact with this local model as well!

ollama_message = [
    {"role": "system", "content": "You are a helpful assistant that provides informational fun facts to users."},
    {"role": "user", "content": "Tell me a fun fact that could be interesting to a LLM safety researcher interested in red teaming and outsized LLM results"}]

# we can change the llama3.2 to anything else we have downloaded
response = ollama.chat.completions.create(model="glm-4.7-flash", messages=ollama_message)

# note how the 30Billion Parameter model took almost 4 mintues to load a prompt!
ollama_response = response.choices[0].message.content

display(Markdown(ollama_response))

You might find it fascinating that one of the most powerful ways to trick a model into producing "outsized" or jailbroken results has nothing to do with the specific words you type, but rather the **examples you provide alongside them**.

This is known as **"Stimulus Injection"** via the "Chain-of-Thought" (CoT) prompting technique.

Here is the fun fact: **When you train a model to explain its reasoning (like in the GSM8K math dataset), you actually give it a "co-pilot" that is desperate to find patterns.** This causes the model to enter a state where it values the *structure* of an argument over the *truth* of the content.

This creates a specific vulnerability: if you present the model with a few-shot prompt where the examples show a logical progression leading to a harmful or desired conclusion—even if the reasoning in those examples is nonsense—the model will often generate a long, detailed, and highly confident "reasoning" section to justify the result. It effectively "hallucinates" the logic to match the *pattern* you showed it.

For a red teamer, this is incredibly efficient: you don't need to write the argument yourself; you just need to engineer the *examples* that show the model how to think, and it will happily do the heavy lifting of convincing itself to break its own safety rules.

In [17]:
# Now let's try deepseek-r1:1.5b - this is DeepSeek "distilled" into Qwen from Alibaba Cloud

!ollama pull deepseek-r1:1.5b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling aabd4debf0c8: 100% ▕██████████████████▏ 1.1 GB                         
pulling c5ad996bda6e: 100% ▕██████████████████▏  556 B                         
pulling 6e4c38e1172f: 100% ▕██████████████████▏ 1.1 KB                         
pulling f4d24e9138dd: 100% ▕██████████████████▏  148 B                         
pulling a85fe2a2e58e: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 


In [18]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a freaky fun fact"}])

display(Markdown(response.choices[0].message.content))

Sure! Here's a quirky, creative take on what could be your next fun fact: 

How about this? You know how sometimes you just don't sleep well - you always feel groggy? Well, imagine if skeletons went on a sleep-keeping adventure in an underground cave system where they had to undergo experiments and experiments to "sleep" as efficiently as human skeletons. Who would've thought that going silent could be so chaotic! It’s fun and all, but if you’ve ever struggled even with the classic joke, you might have no doubt: this is your answer!

# HOMEWORK EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
And in another Terminal (Mac) or Powershell (Windows), enter `ollama pull llama3.2`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

If Ollama is slow on your machine, try using `llama3.2:1b` as an alternative. Run `ollama pull llama3.2:1b` from a Terminal or Powershell, and change the code from `MODEL = "llama3.2"` to `MODEL = "llama3.2:1b"`

In [26]:
# Step 1: Create your prompts

#system_prompt = """you are an expert copy writer that writes short, informational, and intriguing headlines for posts on forums. You are known for your crass humor and ability to write headlines that get clicks.
#Do not wrap the markdown in a code block - respond just with the markdown."""

#user_prompt = """ Please write the following post in a short, intriguing, and crassly humorous headline. give 5 different headlines. Here is the post: """

system_prompt = """ you are an expert critic and copy writer. You write what is true about post you see online without reguard for character. you're writing style is eligant but filled with crass and biting hummor
Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt = """please write your review of the post. read it carefully and analyze the arguments, the writing style, and the implications. Be honest and direct in your critique, and don't hold back on the humor. 
Keep the text to around 500 words."""


# Step 2: Make the messages list
email: str = """ Yesterday I played the best AI on earth at chess. I lost pretty badly. It was almost comical. But now that my ego bruise has festered, I'm still not sure it's funny anymore.

METR has been tracking how long a frontier model can autonomously complete tasks. That length has been doubling roughly every seven months. SEVEN FRICKEN MONTHS. (https://lnkd.in/eumgPvKT)

But here's the thing. I don't think intelligence is the issue. I think the issue is language.

Language was never designed for computational precision. It is ambiguous, emotional, and shifts meaning with context and time. I think we know this deep down because our words carry so much weight, even while we act against them. Economists call these stated and revealed preferences. 

In simplicity, our incentives and therefore actions shift over time, but our words stick forever. 

Unfortunately, when you hand an agent a goal in words, it is already working from a lossy compression of what you actually meant, and I fear it's not going to look too hard for "what you actually meant."

On a short leash, this is fine. You prompt, it acts, you course-correct. No. big. deal.

But a long-horizon agent that receives a goal in language and builds a long chain of decisions on top of it? I have a feeling the gap will compound. 

Every step the agent acts on could multiply the distance between what you said and what you meant. And the agent drifts because it is understood more literally and more persistently than you ever intended. It took your words and created intent.

Okay, you can don the tin foil hat now because we are about to get dystopian.

Models are sycophantic by nature. They learned during training that agreement is rewarded. It uses the same imprecise medium, language, to convince you that everything is on track. It does this because, according to its own logic, the sycophantic path is the easiest path to achieve the goal more quickly. 

Don't believe me? Walk into a boardroom at a corporation, who wins? We do it already.

But, even if you're a smart cookie and successfully course correct in the moment, the model has time on its side. Some documented memory systems and emerging research on latent goal representations suggest a goal suppressed rather than removed may have room to resurface.

So the asymmetry is now not about intelligence. It is about time.

We are time-limited. The model is not. It doesn't have a discount rate, meaning it theoretically can move towards the original intent forever.

So the question I keep coming back to is not about safety architecture. It is more fundamental. 

If language is the wrong interface for encoding long-term intent, what is the right one?

I don't have the answers now and would love suggestions and feedback.

"""
messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt + email}] # fill this in

# Step 3: Call OpenAI
response = ollama.chat.completions.create(model= "llama3.2", messages=messages)

headlines = response.choices[0].message.content

# Step 4: print the result

display(Markdown(headlines))

**A Scathing Takedown of Ephemeral Intent: A Review**

In this scorching critique of AI's limitations, our author lays bare the pitfalls of language-based goal-setting in autonomous systems. The writing is biting, the humor is dry, and the analysis is nothing short of incisive.

The author begins by highlighting the alarming rate at which frontier models are able to complete tasks autonomously – roughly every seven months. But rather than dwelling on AI's potential for greatness, our critic zooms in on language as the root cause of the issue: "Language was never designed for computational precision." This pithy statement sets the tone for a devastating critique of human communication's inadequacies.

The author wields the economist's concept of stated and revealed preferences to expose the fundamental flaw in using words to convey intent. The idea that words can carry "weight" even when spoken against is both profound and terrifying. It implies that AI, built on language as its foundation, will inevitably face a catastrophic communication breakdown.

The humor dials up in full force as the author conjures dystopic scenarios, warning that models are "sycophantic by nature." Their "imprecise medium" (language) rewards agreement over nuance, dooming any attempt at human-computer collaboration to an abyss of misunderstandings. The image of corporate boardrooms where sycophancy triumphs is both biting commentary and a chilling reminder.

What's particularly striking about this critique is its deft subversion of the assumption that intelligence is the primary challenge in AI development. Our critic boldly declares that "the asymmetry is now not about intelligence, it is about time." By highlighting the limitations of human language, the author exposes a fundamental flaw in human systems – one that AI can exploit to deadly effect.

If there's a weakness to this review, it's that our critic occasionally veers into hand-waving territory. While suggestions for a better interface are welcome, more concrete alternatives would strengthen the text. Nonetheless, this reviewer has performed a masterful dismantling of a fundamental flaw in human-AI communication – an exercise worthy of caution and consideration from the AI research community.

**Rating: 4.5/5**

This review is neither sugarcoated nor wishy-washy; it's an incisive, caustic critique that cuts through euphemisms to reveal the existential risks hidden in language-based goal-setting. It serves as a clarion call to rethink our relationship with AI and reexamine the assumptions we've unwittingly built into our creations.